# Extension 1: Churn Prediction

**Goal:** Predict which active users will stop using the app within 30 days.

**Model:** `GBTClassifier` via Spark MLlib `Pipeline`

**Data sources:**
| Table | Role |
|---|---|
| `session_metrics` | Per-session behavioral features |
| `user_metadata` | User profile features |
| `daily_active_users` | Date range reference |

**Feasibility signal:** AUC-ROC > 0.70 confirms meaningful churn signal in existing features.

---
**Prerequisites:** Run `make run-jobs` (or `make quickstart`) before opening this notebook.
All 13 PostgreSQL tables must be populated.

## Cell 1 — Setup

In [ ]:
import os
import pandas as pd
import psycopg2
from pyspark.sql import SparkSession

PG = dict(
    host=os.getenv("POSTGRES_HOST", "postgres"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    dbname=os.getenv("POSTGRES_DB", "analytics"),
    user=os.getenv("POSTGRES_USER", "analytics_user"),
    password=os.getenv("POSTGRES_PASSWORD", "analytics_pass"),
)

def pg_query(sql: str) -> pd.DataFrame:
    """Execute SQL and return a pandas DataFrame."""
    # psycopg2's context manager only manages transactions (commit/rollback),
    # NOT the connection lifecycle — close() must be called explicitly.
    conn = psycopg2.connect(**PG)
    try:
        return pd.read_sql(sql, conn)
    finally:
        conn.close()

# Connect to the existing Spark cluster.
# spark.driver.host must be set to the Jupyter container hostname so that
# spark-worker-1 and spark-worker-2 can open the callback channel to the driver.
spark = (
    SparkSession.builder
    .appName("ML Feasibility — Churn Prediction")
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", "goodnote-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "master:", spark.sparkContext.master)

## Cell 2 — Feature Engineering

Aggregate per-user session behavior and join with profile metadata.
The churn label is derived from recency: users inactive for > 30 days are labelled churned.

In [ ]:
df = pg_query("""
    SELECT
        s.user_id,
        COUNT(*)                                   AS total_sessions,
        AVG(s.session_duration_ms) / 1000.0        AS avg_session_sec,
        AVG(s.actions_count)                       AS avg_actions_per_session,
        SUM(s.is_bounce)::float / COUNT(*)         AS personal_bounce_rate,
        MAX(s.session_start_time)                  AS last_seen,
        m.device_type,
        m.country,
        m.subscription_type
    FROM session_metrics s
    JOIN user_metadata m USING (user_id)
    GROUP BY s.user_id, m.device_type, m.country, m.subscription_type
""")

print(f"Users loaded: {len(df):,}")
df.head()

## Cell 3 — Label Definition

A user is **churned** if their last seen session was more than 30 days before the most recent
session date in the dataset. This is a proxy label derived entirely from existing data.

In [ ]:
CHURN_WINDOW_DAYS = 30

df["last_seen"] = pd.to_datetime(df["last_seen"])
reference_date = df["last_seen"].max()

df["churned"] = (
    (reference_date - df["last_seen"]).dt.days > CHURN_WINDOW_DAYS
).astype(int)

churn_rate = df["churned"].mean()
print(f"Reference date : {reference_date.date()}")
print(f"Churned users  : {df['churned'].sum():,}  ({churn_rate:.1%} of total)")
print(f"Retained users : {(~df['churned'].astype(bool)).sum():,}")

# NOTE: users last active within CHURN_WINDOW_DAYS of the reference date are
# structurally forced into "retained" — their recency alone prevents a churn
# label regardless of actual future behaviour. This is a proxy label; interpret
# AUC results with that in mind.
days_since_last_seen = (reference_date - df["last_seen"]).dt.days
print(f"\nUsers last active within {CHURN_WINDOW_DAYS}d of reference date (forced retained): "
      f"{(days_since_last_seen <= CHURN_WINDOW_DAYS).sum():,}")

df[["total_sessions", "avg_session_sec", "personal_bounce_rate", "churned"]].describe()

### Layman Explanation

Before a model can learn to predict churn, we need to tell it **which users actually churned**. We cannot see the future, so we use a proxy rule: anyone who has not been seen in the app for more than 30 days before the last date in our dataset is labelled as "churned" (1); everyone else is "retained" (0).

Think of it like marking students "absent" after they have missed 30 consecutive days of school — we do not know for certain they dropped out, but the absence pattern is our best available signal.

### Technical Discussion

`reference_date` is set to `df["last_seen"].max()` — the most recent session date in the entire dataset. The churn label is:

```
churned = int((reference_date − last_seen).days > CHURN_WINDOW_DAYS)
```

This is a **right-censored proxy label**. Users who were active within the final 30 days are structurally forced into the "retained" class regardless of whether they will churn in the future. This inflates the retained class and means:

- **AUC-ROC will be optimistically biased** — the model sees an artificially clean boundary.
- The label captures *recency* as a churn signal, which is reasonable but conflates "not yet churned" with "will never churn."

The notebook prints the count of "forced retained" users so the reader can assess the magnitude of this bias.

### Terminology

| Term | Meaning |
|------|---------|
| **Churn** | A user who stops engaging with the product — they have "churned out" of the active user base. |
| **Proxy label** | A substitute target variable derived from observable data when the true outcome (future churn) cannot be directly observed at training time. |
| **Reference date** | The latest date in the dataset, used as the "present" when computing recency. |
| **Right-censored data** | Observations where we know the user was active up to a point, but we do not know what happened afterwards. Recent users appear retained simply because we ran out of data. |
| **Binary classification target** | A label that takes one of two values — here 0 (retained) or 1 (churned) — that the model learns to predict. |
| **Class imbalance** | When one label is far more common than the other. If 90% of users are retained and 10% churned, a naive model that always predicts "retained" achieves 90% accuracy while being useless for detecting churn. |

## Cell 4 — MLlib Pipeline

Pipeline stages:
1. `StringIndexer` (×3) — encode `device_type`, `country`, `subscription_type`
2. `VectorAssembler` — combine numeric + encoded categoricals into a single feature vector
3. `GBTClassifier` — gradient-boosted tree classifier

Note: `StandardScaler` is intentionally omitted — GBT is a tree-based model and is invariant to feature scaling.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Convert pandas → Spark (fill NaN before conversion)
sdf = spark.createDataFrame(
    df[["total_sessions", "avg_session_sec", "avg_actions_per_session",
        "personal_bounce_rate", "device_type", "country",
        "subscription_type", "churned"]]
    .fillna({"avg_session_sec": 0.0, "avg_actions_per_session": 0.0, "personal_bounce_rate": 0.0})
)
# Cache before split so the DataFrame is not recomputed for count() + fit()
sdf.cache()

# Encode categoricals
device_idx  = StringIndexer(inputCol="device_type",       outputCol="device_idx",  handleInvalid="keep")
country_idx = StringIndexer(inputCol="country",           outputCol="country_idx", handleInvalid="keep")
sub_idx     = StringIndexer(inputCol="subscription_type", outputCol="sub_idx",     handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=[
        "total_sessions", "avg_session_sec", "avg_actions_per_session",
        "personal_bounce_rate", "device_idx", "country_idx", "sub_idx",
    ],
    outputCol="features",
)

# StandardScaler removed: GBTClassifier is a tree-based model and is invariant
# to feature scaling. Scaling added cost and forced sparse→dense conversion.
gbt = GBTClassifier(
    labelCol="churned",
    featuresCol="features",
    maxIter=50,
    maxDepth=5,
    seed=42,
)

pipeline = Pipeline(stages=[device_idx, country_idx, sub_idx, assembler, gbt])

train, test = sdf.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train.count():,}   Test rows: {test.count():,}")
print("Fitting pipeline...")
model = pipeline.fit(train)
print("Done.")
sdf.unpersist()  # free training cache; sdf no longer needed after fit


### Layman Explanation

This cell builds and trains the churn prediction model. Think of it as a **factory assembly line**:

1. Text columns (device type, country, subscription) are converted to numbers because models cannot read strings.
2. All input columns are stacked into a single list per user called a feature vector.
3. A gradient-boosted tree model learns from 80% of the users (training set) which patterns predict churn.

The final model is a committee of 50 decision trees that collectively vote on whether a new user looks like someone who will churn.

### Technical Discussion

The `Pipeline` chains four stages:

| Stage | Class | Purpose |
|-------|-------|---------|
| 1 | `StringIndexer` (×3) | Encodes each categorical column as a numeric index sorted by frequency. `handleInvalid="keep"` assigns unseen categories a reserved index rather than throwing an error. |
| 2 | `VectorAssembler` | Concatenates all numeric inputs into a single dense `features` vector required by Spark ML estimators. |
| 3 | `GBTClassifier` | Trains a gradient-boosted ensemble of `maxIter=50` depth-5 regression trees. Each tree fits the residual errors of the previous ensemble using a log-loss objective. |

`StandardScaler` is intentionally omitted: GBT is a tree-based model and is invariant to monotone feature rescaling, so scaling adds overhead without benefit.

The 80/20 split uses `randomSplit([0.8, 0.2], seed=42)` — the fixed seed ensures reproducibility.

### Terminology

| Term | Meaning |
|------|---------|
| **Pipeline** | A Spark ML object that chains multiple transformers and estimators into a single reusable workflow. Calling `.fit()` trains the whole chain end-to-end. |
| **StringIndexer** | Converts a text column into integers. "mobile" → 0, "desktop" → 1, etc., ordered by how often each value appears. |
| **VectorAssembler** | Merges multiple numeric columns into one vector column — the format Spark ML models require as input. |
| **GBTClassifier** | Gradient Boosted Tree Classifier. An ensemble method that builds trees sequentially, where each tree corrects the errors of the previous ones. Usually one of the strongest out-of-the-box classifiers for tabular data. |
| **Gradient boosting** | An iterative ensemble technique: start with a weak prediction, compute the residuals (errors), fit a new tree to those residuals, add it to the ensemble, and repeat. |
| **Hyperparameter** | A setting chosen before training (e.g. `maxIter=50`, `maxDepth=5`) that controls model complexity. It is not learned from data. |
| **Train / test split** | Dividing data into a portion used to train the model and a held-out portion used to evaluate it — so we measure performance on data the model has never seen. |
| **Feature vector** | A single ordered list of numbers representing one observation (one user) that is passed to the model. |
| **Seed** | A number that initialises a random number generator. Using the same seed guarantees identical splits and results across runs. |

## Cell 5 — Evaluation

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

predictions = model.transform(test)
predictions.cache()  # avoid recomputing model inference for each evaluator

auc_evaluator = BinaryClassificationEvaluator(labelCol="churned", metricName="areaUnderROC")
pr_evaluator  = BinaryClassificationEvaluator(labelCol="churned", metricName="areaUnderPR")
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="churned", predictionCol="prediction", metricName="accuracy"
)

auc = auc_evaluator.evaluate(predictions)
apr = pr_evaluator.evaluate(predictions)
acc = acc_evaluator.evaluate(predictions)

print(f"AUC-ROC  : {auc:.4f}   (feasibility threshold: 0.70)")
print(f"AUC-PR   : {apr:.4f}")
print(f"Accuracy : {acc:.4f}")
print()
if auc >= 0.70:
    print("RESULT: FEASIBLE — session + metadata features carry meaningful churn signal.")
    print("Recommended next step: promote to src/jobs/05_ml_churn.py")
else:
    print("RESULT: Further feature engineering needed before productionising.")

### Layman Explanation

After training, we test the model on the 20% of users it has never seen and measure three things:

- **AUC-ROC** — how well can the model rank churners above non-churners? Score of 1.0 is perfect; 0.5 is no better than a coin flip. We need at least 0.70 to consider the model useful.
- **AUC-PR** — similar idea, but focuses on precision and recall. More informative when churned users are a small minority.
- **Accuracy** — the simple percentage of users correctly classified. Can be misleading when one class is rare.

### Technical Discussion

`BinaryClassificationEvaluator` computes both AUC metrics using the model's raw probability scores (`rawPrediction` column), not the hard 0/1 decisions. This means the metric is **threshold-independent** — it summarises performance across all possible decision thresholds.

- **AUC-ROC** integrates the ROC curve (true positive rate vs false positive rate). It answers: "if I pick a random churner and a random non-churner, how often does the model rank the churner higher?"
- **AUC-PR** integrates the precision-recall curve. For imbalanced datasets (e.g. 10% churn rate), AUC-PR is a stricter and more meaningful measure because it penalises false positives heavily.

`MulticlassClassificationEvaluator` with `metricName="accuracy"` counts the fraction of exact prediction matches using the default 0.5 threshold. Its limitation: if 90% of users are retained, a model predicting "retained" for everyone achieves 90% accuracy while being worthless.

### Terminology

| Term | Meaning |
|------|---------|
| **AUC-ROC** | Area Under the Receiver Operating Characteristic curve. Measures the model's ability to distinguish between classes across all classification thresholds. Range: 0 to 1; 0.5 = random. |
| **AUC-PR** | Area Under the Precision-Recall curve. Measures the trade-off between precision (how many predicted churners actually churned) and recall (how many real churners were caught). |
| **Threshold** | The probability cut-off above which the model predicts "churned." Lowering it catches more churners but also generates more false alarms. |
| **True positive rate (Recall / Sensitivity)** | Fraction of actual churners the model correctly identifies. |
| **False positive rate** | Fraction of retained users the model incorrectly flags as churned. |
| **Precision** | Of all users the model predicted as churned, what fraction actually churned? |
| **Class imbalance** | When one label is far more common than the other, making accuracy misleading as a sole metric. |
| **Threshold-independent metric** | A metric (like AUC-ROC) that evaluates the model's ranking ability without committing to a single threshold. |

## Cell 6 — Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Locate the fitted VectorAssembler by type rather than by hardcoded index,
# so this cell remains correct if pipeline stages are ever added or reordered.
assembler_stage = next(s for s in model.stages if isinstance(s, VectorAssembler))
feature_names = assembler_stage.getInputCols()

# GBTClassifier is the last stage
importances = model.stages[-1].featureImportances.toArray()

sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(importances)), importances[sorted_idx])
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([feature_names[i] for i in sorted_idx], rotation=35, ha="right")
ax.set_title("GBT Feature Importances — Churn Prediction")
ax.set_ylabel("Importance")
plt.tight_layout()
plt.show()

### Layman Explanation

After training, we ask the model: **which inputs were most useful for predicting churn?**

The bar chart ranks every feature from most to least important. Features that appear near the top — like total sessions or average session length — are the strongest signals. This tells the product team where to focus: if engagement depth drives churn, fixing session length matters more than, say, optimising for country.

### Technical Discussion

GBT feature importances are computed as the **normalised sum of impurity reductions** across all trees, weighted by the number of samples that flowed through each split:

```
importance(feature f) ∝ Σ_{trees} Σ_{splits on f} (impurity_before − impurity_after) × n_samples
```

The result is a `SparseVector` normalised so all importances sum to 1.0. It is accessed via `model.stages[-1].featureImportances`.

Feature names are retrieved by locating the fitted `VectorAssembler` stage using `isinstance` rather than a hardcoded index. This ensures the chart remains correct if stages are ever added or reordered in the pipeline.

`np.argsort(importances)[::-1]` produces indices that sort the array in descending order, allowing bar positions and labels to align correctly.

### Terminology

| Term | Meaning |
|------|---------|
| **Feature importance** | A score assigned to each input variable indicating how much it contributed to the model's predictions. Higher = more useful. |
| **Impurity reduction** | The decrease in a node's impurity (e.g. Gini impurity or variance) after a split. A large reduction means the split cleanly separates classes. |
| **Gini impurity** | A measure of how mixed the class labels are at a tree node. A value of 0 means the node is pure (all one class); 0.5 means maximally mixed. |
| **Normalised importance** | Importances are rescaled so they sum to 1, enabling relative comparison across features. |
| **SparseVector** | A Spark data type for lists with many zero values, storing only non-zero entries for efficiency. |
| **`np.argsort`** | Returns the indices that would sort an array. Combined with `[::-1]` (reverse), it gives descending order. |

## Cell 7 — Confusion Matrix

In [ ]:
cm = (
    predictions
    .groupBy("churned", "prediction")
    .count()
    .orderBy("churned", "prediction")
    .toPandas()
)
print("Confusion matrix (churned vs prediction):")
print(cm.to_string(index=False))

### Layman Explanation

A confusion matrix is a **2×2 scoreboard** that counts how the model classified each user:

|  | Predicted: Retained | Predicted: Churned |
|--|--|--|
| **Actual: Retained** | Correctly left alone ✓ | False alarm — wasted re-engagement effort |
| **Actual: Churned** | Missed churn — lost user | Correctly flagged ✓ |

The off-diagonal cells are the errors. In a churn context, **missing a churner** (bottom-left) is usually more expensive than **falsely flagging a retained user** (top-right), because a missed churner means a lost customer.

### Technical Discussion

The confusion matrix is computed with a Spark `groupBy` on `(churned, prediction)` followed by `.count()`. This is equivalent to a cross-tabulation. The result has at most four rows (one per cell of the 2×2 matrix).

- `churned=0, prediction=0` → **True Negatives (TN)**: correctly predicted retained.
- `churned=1, prediction=1` → **True Positives (TP)**: correctly predicted churned.
- `churned=0, prediction=1` → **False Positives (FP)**: incorrectly predicted churned (false alarm).
- `churned=1, prediction=0` → **False Negatives (FN)**: incorrectly predicted retained (missed churn).

From these four numbers, all classification metrics can be derived:
- Precision = TP / (TP + FP)
- Recall = TP / (TP + FN)
- F1 = 2 × Precision × Recall / (Precision + Recall)

### Terminology

| Term | Meaning |
|------|---------|
| **Confusion matrix** | A table showing the count of correct and incorrect predictions for each class combination. |
| **True Positive (TP)** | Model correctly predicted the positive class (churned = 1). |
| **True Negative (TN)** | Model correctly predicted the negative class (retained = 0). |
| **False Positive (FP)** | Model predicted churned but user was actually retained. Also called a Type I error. |
| **False Negative (FN)** | Model predicted retained but user actually churned. Also called a Type II error. |
| **Precision** | Of everyone the model predicted as churned, what fraction truly churned? Penalises false alarms. |
| **Recall (Sensitivity)** | Of everyone who actually churned, what fraction did the model catch? Penalises missed churners. |
| **F1 score** | The harmonic mean of precision and recall — a single number balancing both concerns. |

## Cell 8 — Cleanup

In [ ]:
predictions.unpersist()
spark.stop()
print("Spark session stopped.")